# 🎹 Aria Piano Generator
**Model:** EleutherAI Aria (LLaMA 3.2 1B) — trained on 60k hours of solo piano MIDI  
**Paper:** [Scaling Self-Supervised Representation Learning for Symbolic Piano Performance](https://arxiv.org/abs/2506.23869) (ISMIR 2025)  
**Checkpoint used:** `aria-medium-gen` — the generation-finetuned version

### How this works
Aria is a continuation model — it needs a short seed MIDI to "prime" it, then generates a full piece from there.  
We'll use a real Chopin snippet as the seed (first ~15 seconds), then let the model generate 2+ minutes of original music continuing in that style.

**Runtime:** Make sure you have a **T4 GPU** selected (Runtime → Change runtime type → T4 GPU)

---

## Step 1 — Install Dependencies

In [ ]:
# Install Aria and its MIDI tooling
!pip install -q git+https://github.com/EleutherAI/aria-utils.git
!pip install -q transformers safetensors huggingface_hub pretty_midi

print("✅ Dependencies installed")

## Step 2 — Load the Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("⚠️  WARNING: No GPU found. Generation will be very slow. Switch to a T4 runtime.")

print("\nLoading Aria model and tokenizer from HuggingFace...")
print("(First run downloads ~2.8 GB — grab a coffee ☕)")

# Load model architecture + base weights
model = AutoModelForCausalLM.from_pretrained(
    "loubb/aria-medium-base",
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)

# Load the generation-finetuned weights (better musical quality than base)
print("\nDownloading generation-finetuned checkpoint...")
gen_ckpt_path = hf_hub_download(
    repo_id="loubb/aria-medium-base",
    filename="model-gen.safetensors"
)
gen_state_dict = load_file(gen_ckpt_path)
model.load_state_dict(gen_state_dict, strict=False)
model = model.to(DEVICE)
model.eval()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "loubb/aria-medium-base",
    trust_remote_code=True,
)

print("\n✅ Model loaded and ready!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

## Step 3 — Get a Seed MIDI

You have two options:
- **Option A (recommended):** Upload your own Chopin MIDI file
- **Option B:** Download a free Chopin MIDI automatically

**Tip:** The better and more expressive your seed, the better the output. Use a real human performance MIDI, not a score/quantized one.

In [ ]:
import os

# ─────────────────────────────────────────────────────────
# OPTION A: Upload your own MIDI
# ─────────────────────────────────────────────────────────
USE_UPLOAD = False  # Set to True if you want to upload your own file

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    SEED_MIDI_PATH = list(uploaded.keys())[0]
    print(f"Using uploaded file: {SEED_MIDI_PATH}")

# ─────────────────────────────────────────────────────────
# OPTION B: Download a Chopin example from Aria's repo
# ─────────────────────────────────────────────────────────
else:
    print("Downloading example Chopin seed from Aria repository...")
    # Use the Waltz example from Aria's own example-prompts directory
    !wget -q -O seed_chopin.mid "https://raw.githubusercontent.com/EleutherAI/aria/main/example-prompts/waltz.mid"

    # If that fails (file may move), fall back to creating a minimal seed
    if not os.path.exists("seed_chopin.mid") or os.path.getsize("seed_chopin.mid") < 100:
        print("Direct download failed, creating a minimal seed...")
        import pretty_midi
        import numpy as np

        # A minimal Chopin-like waltz seed (Eb major, 3/4 time)
        # Just enough for the model to understand: piano, classical style
        pm = pretty_midi.PrettyMIDI(initial_tempo=144.0)
        piano = pretty_midi.Instrument(program=0)  # Acoustic Grand Piano

        # Simple bass + melody pattern in Eb major for 4 bars
        # Bass notes (beat 1)
        bass = [39, 39, 34, 34]      # Eb3, Eb3, Bb2, Bb2
        # Chord notes (beats 2+3)
        chord_hi = [63, 63, 62, 62]  # Eb4, Eb4, D4, D4
        chord_lo = [55, 55, 54, 54]  # G3, G3, F#3, F#3
        # Melody
        melody = [
            (67, 0.0, 0.4), (67, 0.5, 0.9), (68, 1.0, 1.4), (70, 1.5, 1.9),
            (72, 2.0, 2.4), (70, 2.5, 2.9), (68, 3.0, 3.4), (67, 3.5, 4.9),
            (65, 5.0, 5.4), (65, 5.5, 5.9), (63, 6.0, 6.4), (62, 6.5, 6.9),
            (63, 7.0, 7.4), (62, 7.5, 7.9), (60, 8.0, 8.4), (58, 8.5, 9.9),
        ]
        beat = 60.0 / 144.0
        for i, (b, c_hi, c_lo) in enumerate(zip(bass, chord_hi, chord_lo)):
            t = i * 3 * beat
            piano.notes.append(pretty_midi.Note(velocity=70, pitch=b, start=t, end=t+beat*0.9))
            piano.notes.append(pretty_midi.Note(velocity=60, pitch=c_hi, start=t+beat, end=t+beat*2.9))
            piano.notes.append(pretty_midi.Note(velocity=60, pitch=c_lo, start=t+beat, end=t+beat*2.9))
        for pitch, start, end in melody:
            piano.notes.append(pretty_midi.Note(velocity=80, pitch=pitch,
                                                 start=start*beat, end=end*beat))
        pm.instruments.append(piano)
        pm.write("seed_chopin.mid")
        print("Created minimal seed MIDI.")

    SEED_MIDI_PATH = "seed_chopin.mid"
    print(f"✅ Seed MIDI ready: {SEED_MIDI_PATH}")

## Step 4 — Generate! 🎹

Adjust the settings below to taste:

In [ ]:
# ╔══════════════════════════════════════════╗
# ║          GENERATION SETTINGS            ║
# ╚══════════════════════════════════════════╝

SEED_DURATION_SEC  = 15    # How many seconds of the seed to use as context (8–30 sec)
                           #   More seed = model stays closer to the seed's style
                           #   Less seed = more original but potentially less coherent

MAX_NEW_TOKENS     = 3500  # Tokens to generate (~4000 = ~2 min of piano music)
                           #   1 token ≈ ~30ms of music at typical piano density
                           #   3500 tokens ≈ 1.5–2.5 min depending on note density

TEMPERATURE        = 0.97  # Expressiveness (0.8=safe/repetitive, 1.1=expressive/risky)
                           #   0.95–1.0 is the sweet spot for Chopin style

TOP_P              = 0.95  # Nucleus sampling — keeps generation musical (don't go below 0.9)

NUM_VARIATIONS     = 3     # How many different pieces to generate from the same seed
                           #   Each one will be different due to random sampling

OUTPUT_DIR         = "generated_midi"  # Where to save the output files

# ══════════════════════════════════════════

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"🎹 Generating {NUM_VARIATIONS} variations...")
print(f"   Seed duration : {SEED_DURATION_SEC}s")
print(f"   New tokens    : {MAX_NEW_TOKENS} (~{MAX_NEW_TOKENS * 0.03 / 60:.1f}–{MAX_NEW_TOKENS * 0.05 / 60:.1f} min of music)")
print(f"   Temperature   : {TEMPERATURE}")
print(f"   Top-p         : {TOP_P}")
print()

for i in range(NUM_VARIATIONS):
    print(f"Generating variation {i+1}/{NUM_VARIATIONS}...", end="", flush=True)

    # Tokenize the seed MIDI
    prompt = tokenizer.encode_from_file(
        SEED_MIDI_PATH,
        return_tensors="pt",
        truncate_seconds=SEED_DURATION_SEC,
    )
    input_ids = prompt.input_ids.to(DEVICE)

    seed_len = input_ids.shape[1]

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode to MIDI — optionally strip the seed portion so output is "new" music only
    full_sequence   = output[0].tolist()
    new_only        = output[0][seed_len:].tolist()  # tokens after the seed

    # Decode full sequence (seed + continuation) — recommended, sounds more natural
    midi_dict = tokenizer.decode(full_sequence)
    out_path  = os.path.join(OUTPUT_DIR, f"aria_chopin_variation_{i+1}.mid")
    midi_dict.to_midi().save(out_path)

    print(f" ✅ Saved → {out_path}")

print(f"\n🎉 Done! {NUM_VARIATIONS} MIDI files saved to '{OUTPUT_DIR}/'")

## Step 5 — Listen in Colab

In [ ]:
# Install FluidSynth for playback
!apt-get install -qq fluidsynth
!pip install -q midi2audio

# Download a decent soundfont
!wget -q -O /usr/share/sounds/sf2/MuseScore_General.sf2 \
    "https://ftp.osuosl.org/pub/musescore/soundfont/MuseScore_General/MuseScore_General.sf2"

print("Audio playback ready ✅")

In [ ]:
from midi2audio import FluidSynth
from IPython.display import Audio, display
import glob

SF2_PATH = "/usr/share/sounds/sf2/MuseScore_General.sf2"
fs = FluidSynth(sound_font=SF2_PATH)

midi_files = sorted(glob.glob(f"{OUTPUT_DIR}/*.mid"))

for midi_path in midi_files:
    wav_path = midi_path.replace(".mid", ".wav")
    fs.midi_to_audio(midi_path, wav_path)
    name = os.path.basename(midi_path)
    print(f"\n🎹 {name}")
    display(Audio(wav_path))

## Step 6 — Download Your Files

In [ ]:
import zipfile
from google.colab import files

# Zip all outputs
zip_path = "aria_generated_piano.zip"
with zipfile.ZipFile(zip_path, 'w') as zf:
    for midi_path in sorted(glob.glob(f"{OUTPUT_DIR}/*.mid")):
        zf.write(midi_path)
    for wav_path in sorted(glob.glob(f"{OUTPUT_DIR}/*.wav")):
        zf.write(wav_path)

print(f"Downloading {zip_path}...")
files.download(zip_path)

---
## 💡 Tips for Better Results

**Getting more Chopin-like output:**
- Use a real Chopin performance MIDI as your seed (not a score/quantized MIDI). The model responds to expressive performance data.
- Good free sources: [Kunstderfuge](http://www.kunstderfuge.com/chopin.htm), [Mutopia Project](https://www.mutopiaproject.org/)
- Try nocturnes or mazurkas as seeds — they tend to generate the most coherent continuations

**Parameter tuning:**
- If output sounds **too random / wrong notes**: lower temperature to `0.90`
- If output sounds **repetitive / boring**: raise temperature to `1.05`, or increase `SEED_DURATION_SEC`
- If output **diverges too far from Chopin**: reduce `SEED_DURATION_SEC` to 8–10s and try a different seed

**Generation length:**
- `MAX_NEW_TOKENS = 4096` → roughly 2–3 minutes of music (exact length depends on note density)
- You can go up to `8192` for a longer piece but generation takes much longer on T4

**Note on memorization:**
> The model has seen a lot of Chopin in training, so if you use a well-known piece as seed (e.g. Nocturne Op.9 No.2), it may partially "remember" it. For more original output, seed with a lesser-known Chopin piece or your own composition.